# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np
from collections import defaultdict

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Coordenadas tomadas de la imagen
        self.start = (0, 0)

        self.walls = {
            (0, 3),
            (1, 1),
            (2, 4),
            (4, 2),
        }

        self.slippery_states = {
            (1, 2),
            (2, 1),
            (3, 3),
        }

        self.terminal_states = {
            (0, 5): 10.0,   # entrega
            (2, 2):  2.0,   # estación de carga
            (3, 5): -10.0,  # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state

        return (
            0 <= r < self.height
            and 0 <= c < self.width
            and state not in self.walls
        )

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if self.is_valid_state((r, c))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s)
        if state in self.terminal_states:
            return self.terminal_states[state]

        if state in self.danger_states:
            return self.danger_states[state]

        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Las probabilidades dependen de si el estado actual
        es normal o resbaloso.
        """

        # Un estado terminal es absorbente
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_forward = 0.60
            p_left = 0.20
            p_right = 0.20
        else:
            p_forward = 0.90
            p_left = 0.05
            p_right = 0.05

        dr, dc = action

        # Rotaciones respecto a la acción elegida
        left_action = (-dc, dr)
        right_action = (dc, -dr)

        possible_moves = [
            (action, p_forward),
            (left_action, p_left),
            (right_action, p_right),
        ]

        transitions = defaultdict(float)

        for move, probability in possible_moves:
            next_state = (
                state[0] + move[0],
                state[1] + move[1],
            )

            # Si golpea un borde o una pared, se queda quieto
            if not self.is_valid_state(next_state):
                next_state = state

            transitions[next_state] += probability

        return list(transitions.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(
        probability * V[next_state]
        for next_state, probability
        in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        delta = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                best_future_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )

                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_future_value
                )

            delta = max(
                delta,
                abs(V_new[state] - V[state])
            )

        V = V_new

        if delta < threshold:
            return V, iteration + 1

    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            continue

        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(
                grid, state, action, V
            )
        )

    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        delta = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]

                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma
                    * expected_next_value(grid, state, action, V)
                )

            delta = max(
                delta,
                abs(V_new[state] - V[state])
            )

        V = V_new

        if delta < threshold:
            return V, iteration + 1

    return V, max_iter


def policy_improvement(grid, V):
    new_policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            continue

        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(
                grid, state, action, V
            )
        )

    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # Política inicial arbitraria
    policy = {
        state: grid.actions[0]
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []

    for iteration in range(max_iter):
        # 1. Evaluar la política actual
        V, evaluation_iterations = policy_evaluation(
            grid, policy, threshold
        )

        # 2. Mejorar la política
        new_policy = policy_improvement(grid, V)

        changed_actions = sum(
            new_policy[state] != policy[state]
            for state in new_policy
        )

        history.append({
            "iteración": iteration + 1,
            "evaluación": evaluation_iterations,
            "acciones_cambiadas": changed_actions,
        })

        policy = new_policy

        # 3. Si no cambia, la política es estable
        if changed_actions == 0:
            break

    # Evaluación final de la política estable
    V, _ = policy_evaluation(grid, policy, threshold)

    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'iteración': 1, 'evaluación': 89, 'acciones_cambiadas': 16}, {'iteración': 2, 'evaluación': 77, 'acciones_cambiadas': 5}, {'iteración': 3, 'evaluación': 22, 'acciones_cambiadas': 1}, {'iteración': 4, 'evaluación': 22, 'acciones_cambiadas': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770

## Parte 5 — Interpreta la política

1. **Desde `START`, ¿el robot busca la entrega +10 o prefiere la estación de carga +2?**  
   Con los parámetros originales, la política óptima desde `START` se dirige hacia la **estación de carga +2**.

2. **¿Por qué una recompensa menor podría ser óptima?**  
   Porque no importa solamente la recompensa final. También influyen la distancia, el costo por paso, el descuento y el riesgo de desviarse. Una recompensa menor pero más cercana y segura puede tener mayor valor esperado.

3. **¿En qué estados el piso resbaloso cambia la decisión?**  
   En los estados resbalosos `(1,2)`, `(2,1)` y `(3,3)` la acción puede cambiar porque la probabilidad de avanzar baja de `0.90` a `0.60`, mientras que las desviaciones aumentan a `0.20`.

4. **¿Qué papel cumple el costo por paso `-1`?**  
   Penaliza recorridos largos. Así, el robot tiene incentivo para llegar a un terminal sin dar vueltas innecesarias.

5. **¿Por qué `T(s,a,s')` ya no puede usar las mismas probabilidades para todos los estados?**  
   Porque la dinámica depende del estado actual: en piso normal se usa `0.90/0.05/0.05`, mientras que en piso resbaloso se usa `0.60/0.20/0.20`.

### Experimento A — Menos costo por paso

Con `living_reward = -0.1` espero que los caminos largos sean menos castigados, por lo que llegar a `+10` se vuelve relativamente más atractivo.

### Experimento B — Piso muy resbaloso

Si la probabilidad deseada baja de `0.60` a `0.40`, el robot debe dar todavía más importancia al riesgo de desviación, especialmente cerca de peligros y terminales negativos.

### Experimento C — Más paciencia

Con `gamma = 0.99`, las recompensas futuras pierden menos valor. Por eso la recompensa `+10`, aunque esté más lejos, se vuelve más importante.

### Bonus

El punto de cambio puede encontrarse probando distintos valores de `living_reward` y observando cuándo cambia la primera acción de la política desde `START`.


## Experimentos

In [7]:
# Experimento A: living_reward = -0.1
grid_a = WarehouseMDP()
grid_a.living_reward = -0.1

V_a, _ = value_iteration(grid_a)
pi_a = extract_policy(grid_a, V_a)

print("=== EXPERIMENTO A ===")
print("living_reward =", grid_a.living_reward)
print("Primera acción desde START:", ARROWS[pi_a[grid_a.start]])
print_policy(grid_a, pi_a)


=== EXPERIMENTO A ===
living_reward = -0.1
Primera acción desde START: →
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


In [8]:
# Experimento B: piso más resbaloso
class WarehouseMDPVerySlippery(WarehouseMDP):
    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_forward = 0.40
            p_left = 0.30
            p_right = 0.30
        else:
            p_forward = 0.90
            p_left = 0.05
            p_right = 0.05

        dr, dc = action
        left_action = (-dc, dr)
        right_action = (dc, -dr)

        possible_moves = [
            (action, p_forward),
            (left_action, p_left),
            (right_action, p_right),
        ]

        transitions = defaultdict(float)

        for move, probability in possible_moves:
            next_state = (
                state[0] + move[0],
                state[1] + move[1],
            )

            if not self.is_valid_state(next_state):
                next_state = state

            transitions[next_state] += probability

        return list(transitions.items())


grid_b = WarehouseMDPVerySlippery()

V_b, _ = value_iteration(grid_b)
pi_b = extract_policy(grid_b, V_b)

print("=== EXPERIMENTO B ===")
print("Piso resbaloso: 0.40 / 0.30 / 0.30")
print("Primera acción desde START:", ARROWS[pi_b[grid_b.start]])
print_policy(grid_b, pi_b)


=== EXPERIMENTO B ===
Piso resbaloso: 0.40 / 0.30 / 0.30
Primera acción desde START: →
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


In [9]:
# Experimento C: gamma = 0.99
grid_c = WarehouseMDP()
grid_c.gamma = 0.99

V_c, _ = value_iteration(grid_c)
pi_c = extract_policy(grid_c, V_c)

print("=== EXPERIMENTO C ===")
print("gamma =", grid_c.gamma)
print("Primera acción desde START:", ARROWS[pi_c[grid_c.start]])
print_policy(grid_c, pi_c)


=== EXPERIMENTO C ===
gamma = 0.99
Primera acción desde START: →
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


In [10]:
# Bonus: buscar aproximadamente dónde cambia la acción desde START
results = []

for living_reward in np.arange(-2.0, 0.51, 0.05):
    test_grid = WarehouseMDP()
    test_grid.living_reward = float(living_reward)

    V_test, _ = value_iteration(test_grid)
    pi_test = extract_policy(test_grid, V_test)

    results.append(
        (living_reward, pi_test[test_grid.start])
    )

changes = []

for i in range(1, len(results)):
    if results[i][1] != results[i - 1][1]:
        changes.append(
            (
                results[i - 1][0],
                ARROWS[results[i - 1][1]],
                results[i][0],
                ARROWS[results[i][1]],
            )
        )

print("=== BONUS ===")

if changes:
    print("Cambios aproximados de la primera acción desde START:")
    for change in changes:
        print(
            f"entre living_reward={change[0]:.2f} ({change[1]}) "
            f"y {change[2]:.2f} ({change[3]})"
        )
else:
    print(
        "No cambió la primera acción desde START "
        "en el rango probado."
    )


=== BONUS ===
No cambió la primera acción desde START en el rango probado.
